# Autonomous Sumo Robot Digital Twin - Stage 3: Strategy Analytics & Spatial Evaluation
## Statistical Micro-Metrics, Spatial Density Heatmaps, and 2D Match Replays

This research notebook analyzes tactical match dynamics using ground-truth Observer Telemetry (`data/observer/*_observer.parquet`).

### Objectives:
1. Ingest observer dataset containing analytical spatial metrics.
2. Compute quantitative micro-combat KPIs:
   - **Time-to-Acquisition (TTA):** Milliseconds elapsed before initial optical target lock.
   - **Center-Control %:** Percentage of match duration occupying the inner ring ($r \le R_{dohyo} / 2$).
   - **Time-to-Ring-Out (TTRO):** Average duration of victorious matches.
3. Plot 2D spatial ring occupancy density heatmaps on Dohyo geometry.
4. Render top-down 2D match replays with true robot hitboxes, bumper indicators, and heading vectors.

In [2]:
from IPython.display import display, Markdown, HTML
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns

# Ensure project root is in Python path
project_root = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from config.config import OBSERVER_DATA_DIR, KIT_1KG_CONFIG, MEGA_3KG_CONFIG, PROJECT_ROOT
from data_pipeline.etl_pipeline import compute_observer_kpis

print("Analytics libraries loaded successfully.")

Analytics libraries loaded successfully.


### 1. Load Observer Telemetry

In [4]:
# Auto-discover observer dataset or fallback to standard path
obs_files = sorted(list(OBSERVER_DATA_DIR.glob("*_observer.parquet")))
if obs_files:
    obs_parquet_path = obs_files[0]
else:
    obs_parquet_path = OBSERVER_DATA_DIR / "telemetry_kit_1kg_observer.parquet"

df_obs = pd.read_parquet(obs_parquet_path)
print(f"Loaded {len(df_obs):,} observer records across {df_obs['Match_ID'].nunique()} matches from {obs_parquet_path.name}.")
df_obs.head()

Loaded 114,060 observer records across 50 matches from telemetry_kit_1kg_observer.parquet.
Out[0]: 
  Match_ID  Tick  Timestamp_ms  ... Target_Visible Current_State Match_Status
0  M_00001     0             0  ...           True        ATTACK  IN_PROGRESS
1  M_00001     0             0  ...          False         TRACK  IN_PROGRESS
2  M_00001     1            50  ...           True        ATTACK  IN_PROGRESS
3  M_00001     1            50  ...           True         TRACK  IN_PROGRESS
4  M_00001     2           100  ...           True        ATTACK  IN_PROGRESS

[5 rows x 14 columns]


  Match_ID  Tick  Timestamp_ms  ... Target_Visible Current_State Match_Status
0  M_00001     0             0  ...           True        ATTACK  IN_PROGRESS
1  M_00001     0             0  ...          False         TRACK  IN_PROGRESS
2  M_00001     1            50  ...           True        ATTACK  IN_PROGRESS
3  M_00001     1            50  ...           True         TRACK  IN_PROGRESS
4  M_00001     2           100  ...           True        ATTACK  IN_PROGRESS

[5 rows x 14 columns]

### 2. Micro-Combat KPI Calculations

In [6]:
# Detect weight class configuration dynamically
robot_class = df_obs['Weight_Class'].iloc[0] if 'Weight_Class' in df_obs.columns else ('MEGA_3KG' if '3kg' in obs_parquet_path.name.lower() or 'mega' in obs_parquet_path.name.lower() else 'KIT_1KG')
config = MEGA_3KG_CONFIG if robot_class == 'MEGA_3KG' else KIT_1KG_CONFIG
print(f"Using Configuration: {config.class_name} (Arena Radius: {config.dohyo_radius_cm} cm)")
center_zone_radius = config.center_zone_radius_cm

# Compute micro-combat KPIs using fast vectorized ETL pipeline function
df_kpis = compute_observer_kpis(df_obs, center_zone_r=center_zone_radius)

print("================ MICRO-COMBAT PERFORMANCE SUMMARY ================")
print(f"Mean Time-to-Acquisition (TTA):      {df_kpis['TTA_ms'].mean():.1f} ms (+/- {df_kpis['TTA_ms'].std():.1f} ms)")
print(f"Mean Center-Control Percentage:       {df_kpis['Center_Control_Pct'].mean():.2f} % (+/- {df_kpis['Center_Control_Pct'].std():.2f} %)")
print(f"Mean Time-to-Ring-Out (TTRO):         {df_kpis['TTRO_ms'].dropna().mean():.1f} ms (+/- {df_kpis['TTRO_ms'].dropna().std():.1f} ms)")
print(f"Win Rate:                            {(df_kpis['Status'] == 'BOT_A_WIN').mean() * 100:.2f} %")
df_kpis.head(10)


Using Configuration: KIT_1KG (Arena Radius: 38.5 cm)
================ MICRO-COMBAT PERFORMANCE SUMMARY ================
Mean Time-to-Acquisition (TTA):      52.0 ms (+/- 175.0 ms)
Mean Center-Control Percentage:       73.30 % (+/- 27.90 %)
Mean Time-to-Ring-Out (TTRO):         2517.6 ms (+/- 3355.4 ms)
Win Rate:                            34.00 %
Out[0]: 
  Match_ID  Tick  ...          Strategy_B     Formation_B
0  M_00001    20  ...  AGGRESSIVE_CHARGER      SIDE_START
1  M_00002   258  ...     JUGGERNAUT_PUSH         HEAD_ON
2  M_00003  3599  ...  AGGRESSIVE_CHARGER   ANGLED_INWARD
3  M_00004  3599  ...  AGGRESSIVE_CHARGER      SIDE_START
4  M_00005  3599  ...     JUGGERNAUT_PUSH         HEAD_ON
5  M_00006    21  ...     JUGGERNAUT_PUSH      SIDE_START
6  M_00007    22  ...  AGGRESSIVE_CHARGER   ANGLED_INWARD
7  M_00008    17  ...  AGGRESSIVE_CHARGER  LATERAL_OFFSET
8  M_00009  3599  ...   DEFENSIVE_SWEEPER   ANGLED_INWARD
9  M_00010  3599  ...     JUGGERNAUT_PUSH  LATERAL_OFFSET

[10

  Match_ID  Tick  ...          Strategy_B     Formation_B
0  M_00001    20  ...  AGGRESSIVE_CHARGER      SIDE_START
1  M_00002   258  ...     JUGGERNAUT_PUSH         HEAD_ON
2  M_00003  3599  ...  AGGRESSIVE_CHARGER   ANGLED_INWARD
3  M_00004  3599  ...  AGGRESSIVE_CHARGER      SIDE_START
4  M_00005  3599  ...     JUGGERNAUT_PUSH         HEAD_ON
5  M_00006    21  ...     JUGGERNAUT_PUSH      SIDE_START
6  M_00007    22  ...  AGGRESSIVE_CHARGER   ANGLED_INWARD
7  M_00008    17  ...  AGGRESSIVE_CHARGER  LATERAL_OFFSET
8  M_00009  3599  ...   DEFENSIVE_SWEEPER   ANGLED_INWARD
9  M_00010  3599  ...     JUGGERNAUT_PUSH  LATERAL_OFFSET

[10 rows x 26 columns]

### 3. 2D Spatial Positioning Density Heatmap

In [8]:
fig, ax = plt.subplots(figsize=(9, 9))

# Draw Dohyo Outer White Border
outer_circle = plt.Circle((0, 0), config.dohyo_radius_cm, color="#f0f0f0", ec="black", lw=2, zorder=1)
ax.add_patch(outer_circle)

# Draw Black Arena Surface
inner_circle = plt.Circle((0, 0), config.inner_ring_radius_cm, color="#222222", zorder=2)
ax.add_patch(inner_circle)

# Draw Center-Control Zone (Inner 50% radius)
center_circle = plt.Circle((0, 0), config.center_zone_radius_cm, color="none", ec="cyan", lw=1.5, ls="--", zorder=3)
ax.add_patch(center_circle)

# Filter Bot A positions
df_a_all = df_obs[df_obs["Bot_ID"] == "Bot_A"]

# 2D KDE Heatmap
sns.kdeplot(
    x=df_a_all["Pos_X"],
    y=df_a_all["Pos_Y"],
    ax=ax,
    cmap="inferno",
    fill=True,
    alpha=0.65,
    levels=15,
    zorder=4,
)

ax.set_xlim(-config.dohyo_radius_cm - 5, config.dohyo_radius_cm + 5)
ax.set_ylim(-config.dohyo_radius_cm - 5, config.dohyo_radius_cm + 5)
ax.set_aspect("equal")
ax.set_title(f"{config.class_name}: 2D Spatial Occupancy Density Heatmap", fontsize=15)
ax.set_xlabel("X Position (cm)")
ax.set_ylabel("Y Position (cm)")
plt.tight_layout()
plt.show()

<ipython-input-1-7e650f8e5a01>:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 4. Top-Down 2D Match Replay Rendering

In [10]:
def render_top_down_frame(df_match: pd.DataFrame, tick: int, config=None):
    """Render a single 2D top-down snapshot of a match at a specific tick."""
    if config is None:
        robot_cls = df_match["Weight_Class"].iloc[0] if "Weight_Class" in df_match.columns else "KIT_1KG"
        config = MEGA_3KG_CONFIG if ("3KG" in str(robot_cls).upper() or "MEGA" in str(robot_cls).upper()) else KIT_1KG_CONFIG
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # Dohyo geometry
    outer = plt.Circle((0, 0), config.dohyo_radius_cm, color="#e8e8e8", ec="#333333", lw=2, zorder=1)
    inner = plt.Circle((0, 0), config.inner_ring_radius_cm, color="#1a1a1a", zorder=2)
    center = plt.Circle((0, 0), config.center_zone_radius_cm, color="none", ec="#555555", ls=":", zorder=3)
    ax.add_patch(outer)
    ax.add_patch(inner)
    ax.add_patch(center)
    
    # Shikiri start lines
    shik_sep = config.shikiri_separation_cm / 2.0
    shik_len = config.shikiri_length_cm / 2.0
    ax.plot([-shik_sep, -shik_sep], [-shik_len, shik_len], color="brown", lw=3, zorder=3)
    ax.plot([shik_sep, shik_sep], [-shik_len, shik_len], color="brown", lw=3, zorder=3)
    
    # Render Robots at this tick
    df_tick = df_match[df_match["Tick"] == tick]
    colors = {"Bot_A": "#00d2ff", "Bot_B": "#ff4b4b"}
    
    for _, bot_row in df_tick.iterrows():
        bot_id = bot_row["Bot_ID"]
        bx, by = bot_row["Pos_X"], bot_row["Pos_Y"]
        b_deg = bot_row["Heading_Deg"]
        b_rad = np.radians(b_deg)
        
        # Robot Box (Rotated)
        w = config.robot_length_cm
        h = config.robot_width_cm
        t = plt.matplotlib.transforms.Affine2D().rotate_deg(b_deg).translate(bx, by) + ax.transData
        rect_patch = patches.Rectangle((-w/2, -h/2), w, h, facecolor=colors.get(bot_id, "gray"), edgecolor="white", lw=1.5, alpha=0.85, zorder=5, transform=t)
        ax.add_patch(rect_patch)
        
        # Front bumper (Cyan/Lime thick line on front edge +X)
        front_local = [w/2, 0]
        fx = bx + (w/2) * np.cos(b_rad)
        fy = by + (w/2) * np.sin(b_rad)
        ax.plot([bx, fx], [by, fy], color="yellow", lw=2.5, zorder=6)
        ax.scatter([fx], [fy], color="lime", s=40, zorder=7)
        
    ax.set_xlim(-config.dohyo_radius_cm - 5, config.dohyo_radius_cm + 5)
    ax.set_ylim(-config.dohyo_radius_cm - 5, config.dohyo_radius_cm + 5)
    ax.set_aspect("equal")
    ax.set_title(f"Top-Down Match Replay | Match: {df_tick['Match_ID'].iloc[0]} | Tick: {tick} ({tick*config.dt:.2f}s)", fontsize=13)
    plt.show()

# Render frame at tick 20 of the first match
target_match_id = df_obs["Match_ID"].unique()[0] if not df_obs.empty else "M_00001"
m1_df = df_obs[df_obs["Match_ID"] == target_match_id]
if not m1_df.empty:
    render_top_down_frame(m1_df, tick=min(20, int(m1_df['Tick'].max())), config=config)
else:
    print(f'Match {target_match_id} not in observer records.')

<ipython-input-1-70a330bfb662>:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
